# Experiment 1: Base Performance (AL)

Instance-level analysis: compile rate, pass rate per model across base entries.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import load_all_results, load_aggregate_results

# Configure your setups here after running experiments
SETUPS: list[str] = []  # e.g. ["copilot-opus-4-6", "claude-code"]
SETUP_LABELS: dict[str, str] = {}  # e.g. {"copilot-opus-4-6": "Copilot"}

all_results = load_aggregate_results(category="bug-fix")
print(f"Loaded {len(all_results)} setups: {list(all_results.keys())}")

In [ ]:
import pandas as pd
import plotly.express as px

summary_rows = []
for setup, df in all_results.items():
    if SETUPS and setup not in SETUPS:
        continue
    label = SETUP_LABELS.get(setup, setup)
    n_runs = df["run_id"].nunique()
    n_instances = df["instance_id"].nunique()
    compile_rate = df["build"].mean() * 100
    pass_rate = df["resolved"].mean() * 100
    summary_rows.append({
        "Model": label,
        "Runs": n_runs,
        "Instances": n_instances,
        "Compile Rate (%)": round(compile_rate, 1),
        "Pass Rate (%)": round(pass_rate, 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# Bar chart: compile rate and pass rate per model
if not summary_df.empty:
    melted = summary_df.melt(
        id_vars=["Model"],
        value_vars=["Compile Rate (%)", "Pass Rate (%)"],
        var_name="Metric",
        value_name="Rate",
    )
    fig = px.bar(
        melted, x="Model", y="Rate", color="Metric",
        barmode="group", title="AL Base Performance by Model",
        text_auto=True,
    )
    fig.update_layout(yaxis_range=[0, 100])
    fig.show()

In [ ]:
# Per-instance resolution heatmap (instances x models)
if not summary_df.empty:
    frames = []
    for setup, df in all_results.items():
        if SETUPS and setup not in SETUPS:
            continue
        label = SETUP_LABELS.get(setup, setup)
        per_instance = df.groupby("instance_id")["resolved"].mean().reset_index()
        per_instance["Model"] = label
        frames.append(per_instance)

    if frames:
        combined = pd.concat(frames)
        pivot = combined.pivot(index="instance_id", columns="Model", values="resolved")
        fig = px.imshow(
            pivot, aspect="auto",
            title="Per-Instance Resolution Rate",
            labels={"color": "Resolved %"},
        )
        fig.show()